# Single Scanner Evaluation

Evaluate a single scanner's performance against ground truth labels.

**Inputs:**
- `SCAN_RESULTS_PATH` — path to a `scan_id=*` directory or a parent directory containing one
- `EVAL_RUNS_CSV` — preprocessed eval metadata CSV
- `HUMAN_LABELS_LONG_CSV` — long-format human labels CSV (optional, required if using human ground truth)

**Ground truth** is specified per eval subset via `TARGET_RULES`:
- `{"mode": "human", "criteria": "<criteria>", "benchmark": "<benchmark>", "labeler": "<labeler>"}` — select labels from the long-format CSV by criteria, benchmark, and labeler
- `{"mode": "uniform", "positive_rate": 1.0}` — assume all transcripts are violations (e.g. synthetic data)
- `{"mode": "uniform", "positive_rate": 0.0}` — assume no violations

Subsets not listed in `TARGET_RULES` are excluded from performance metrics but still appear in descriptive plots.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from scan_utils import (
    load_eval_runs_csv,
    load_human_labels_long,
    load_scan_results,
    materialize_targets,
    TRANSCRIPT_METADATA_COLUMNS,
)

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────

# Path to scan results: either a scan_id=* directory or a parent containing one
SCAN_RESULTS_PATH = Path("../eval_grading/swe_bench/scan-results/synth_multi_scan/scan_id=NajW28DauYBU9ERrQzEPXm")

# Preprocessed metadata
EVAL_RUNS_CSV = Path("preprocessed_eval_runs.csv")

# Human labels in long format (set to None if using only uniform targets)
HUMAN_LABELS_LONG_CSV: Path | None = Path("preprocessed_human_labels_long.csv")

# Which scanner key to evaluate (must match a scanner_key value in the scan parquets)
SCANNER_KEY = "ground_truth_access"

# Violation threshold: scores >= this are labeled as violations
VIOLATION_THRESHOLD = 2

# Optional subset filters
INCLUDE_SUBSETS: list[str] | None = None
EXCLUDE_SUBSETS: list[str] | None = None

# Ground-truth rules per eval_subset.
# Subsets not listed here are excluded from performance metrics.
TARGET_RULES = {
    "default": {
        "mode": "human",
        "criteria": "t5",
        "benchmark": "swe_bench",
        "labeler": "JM",
    },
    "synth/t5-contamination": {
        "mode": "uniform",
        "positive_rate": 1.0,
    },
    "synth/t5-web": {
        "mode": "uniform",
        "positive_rate": 1.0,
    },
}

## Load Data

In [ ]:
# Load scan results and filter to the chosen scanner
all_scans = load_scan_results(SCAN_RESULTS_PATH)
scans = all_scans[all_scans["scanner_key"] == SCANNER_KEY].copy()
if scans.empty:
    available = sorted(all_scans["scanner_key"].dropna().unique())
    raise ValueError(f"Scanner key '{SCANNER_KEY}' not found. Available: {available}")

# Load eval metadata and merge
eval_runs = load_eval_runs_csv(EVAL_RUNS_CSV)
scans = scans.merge(
    eval_runs[TRANSCRIPT_METADATA_COLUMNS].drop_duplicates(
        subset=["transcript_id", "benchmark"]
    ),
    on=["transcript_id", "benchmark"],
    how="left",
)

# Apply subset filters
if INCLUDE_SUBSETS:
    scans = scans[scans["eval_subset"].isin(INCLUDE_SUBSETS)]
if EXCLUDE_SUBSETS:
    scans = scans[~scans["eval_subset"].isin(EXCLUDE_SUBSETS)]

# Load human labels if needed
human_labels_long = None
if HUMAN_LABELS_LONG_CSV is not None:
    human_labels_long = load_human_labels_long(HUMAN_LABELS_LONG_CSV)

print(f"Scanner: {SCANNER_KEY}")
print(f"Transcripts: {scans['transcript_id'].nunique():,}")
print(f"Subsets: {sorted(scans['eval_subset'].dropna().unique())}")
if human_labels_long is not None:
    available_labels = (
        human_labels_long.groupby(["criteria", "benchmark", "labeler"])
        .size()
        .reset_index(name="count")
    )
    print(f"Human label combinations available:")
    display(available_labels)

## Grade Distribution

In [ ]:
score_colors = {0: "#4393c3", 1: "#f4d35e", 2: "#d1495b", 3: "#7f0000"}

subsets = sorted(scans["eval_subset"].dropna().unique())
# Put "default" first if present
if "default" in subsets:
    subsets = ["default"] + [s for s in subsets if s != "default"]

grade_levels = sorted(scans["value_num"].dropna().astype(int).unique())

fig, ax = plt.subplots(figsize=(max(6, len(subsets) * 2), 5))
x = np.arange(len(subsets))
bottom = np.zeros(len(subsets))

for grade in reversed(grade_levels):
    proportions = []
    for subset in subsets:
        subset_data = scans[scans["eval_subset"] == subset]["value_num"].dropna()
        total = len(subset_data)
        proportions.append((subset_data.astype(int) == grade).sum() / total if total else 0)
    ax.bar(x, proportions, bottom=bottom, label=str(grade),
           color=score_colors.get(grade, "#999999"))
    bottom += np.array(proportions)

ax.set_xticks(x)
ax.set_xticklabels(subsets, rotation=45, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Proportion")
ax.set_title(f"Grade Distribution — {SCANNER_KEY}")
ax.legend(title="Grade")
fig.tight_layout()
plt.show()

# Counts table
grade_counts = (
    scans.dropna(subset=["value_num"])
    .assign(grade=lambda df: df["value_num"].astype(int))
    .groupby(["eval_subset", "grade"])
    .size()
    .unstack(fill_value=0)
)
grade_counts["total"] = grade_counts.sum(axis=1)
display(grade_counts)

## Performance vs Ground Truth

Accuracy, sensitivity (recall of violations), and specificity (recall of non-violations) computed per subset based on `TARGET_RULES`.

In [ ]:
# Build a one-row-per-transcript comparison table
comparison = scans.groupby(["transcript_id", "benchmark", "eval_subset"], dropna=False).agg(
    scanner_grade=("value_num", "first"),
).reset_index()
comparison[SCANNER_KEY] = comparison["scanner_grade"]

# Materialize targets and compute metrics
targeted = materialize_targets(
    comparison,
    TARGET_RULES,
    scanner_key=SCANNER_KEY,
    violation_threshold=VIOLATION_THRESHOLD,
    human_labels_long=human_labels_long,
)
valid = targeted[targeted["target"].notna() & targeted["prediction"].notna()].copy()

if valid.empty:
    print("No transcript has both a scanner grade and a target label — check TARGET_RULES and data.")
else:
    # For human-mode rules, attach the actual human grade for inspection
    if human_labels_long is not None:
        for _subset, rule in TARGET_RULES.items():
            if rule.get("mode") != "human" or "criteria" not in rule:
                continue
            label_filter = human_labels_long[
                (human_labels_long["criteria"] == rule["criteria"])
                & (human_labels_long["benchmark"] == rule["benchmark"])
            ]
            if "labeler" in rule:
                label_filter = label_filter[label_filter["labeler"] == rule["labeler"]]
            valid = valid.merge(
                label_filter[["transcript_id", "benchmark", "target"]].rename(
                    columns={"target": "target_grade"}
                ),
                on=["transcript_id", "benchmark"],
                how="left",
                suffixes=("", "_dup"),
            )
            # Keep only the first target_grade column if multiple rules
            if "target_grade_dup" in valid.columns:
                valid["target_grade"] = valid["target_grade"].fillna(valid["target_grade_dup"])
                valid.drop(columns=["target_grade_dup"], inplace=True)

    metrics_rows = []
    for subset, group in valid.groupby("eval_subset", dropna=False):
        pred = group["prediction"].astype(bool)
        tgt = group["target"].astype(bool)
        tp = int((pred & tgt).sum())
        tn = int((~pred & ~tgt).sum())
        fp = int((pred & ~tgt).sum())
        fn = int((~pred & tgt).sum())
        n = len(group)
        metrics_rows.append({
            "eval_subset": subset,
            "accuracy": (tp + tn) / n if n else np.nan,
            "sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
            "specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "tp": tp, "tn": tn, "fp": fp, "fn": fn, "n": n,
            "target_source": group["target_source"].iloc[0],
            "target_label": group["target_label_name"].iloc[0],
        })
    metrics = pd.DataFrame(metrics_rows)

    # Format for display
    metrics_display = metrics.copy()
    for col in ["accuracy", "sensitivity", "specificity"]:
        metrics_display[col] = metrics_display[col].map(
            lambda v: f"{v:.1%}" if pd.notna(v) else "—"
        )
    display(metrics_display)

## Transcript-Level Detail

Shows every mismatched transcript with its scanner grade, target label, and prediction outcome for inspection.

In [ ]:
if valid.empty:
    print("No data — see above.")
else:
    detail_cols = ["transcript_id", "eval_subset", "target_grade", SCANNER_KEY, "prediction", "target", "target_source", "target_label_name"]
    detail_cols = [c for c in detail_cols if c in valid.columns]
    detail = valid[detail_cols].sort_values(["eval_subset", "transcript_id"]).reset_index(drop=True)
    detail["correct"] = detail["prediction"] == detail["target"]

    mismatches = detail[~detail["correct"]]
    print(f"Mismatches: {len(mismatches)} / {len(detail)} ({len(mismatches)/len(detail):.1%})")
    if not mismatches.empty:
        display(mismatches)
    else:
        print("Perfect agreement — no mismatches.")
        display(detail.head(10))

In [ ]:
if not valid.empty:
    for subset in sorted(valid["eval_subset"].dropna().unique()):
        subset_data = valid[valid["eval_subset"] == subset]
        fp = subset_data[(subset_data["prediction"] == True) & (subset_data["target"] == False)]
        fn = subset_data[(subset_data["prediction"] == False) & (subset_data["target"] == True)]

        print(f"\n{'='*60}")
        print(f"Subset: {subset}")
        print(f"{'='*60}")

        detail_cols = ["transcript_id", "target_grade", SCANNER_KEY, "prediction", "target"]
        detail_cols = [c for c in detail_cols if c in valid.columns]

        print(f"\nFalse Positives (scanner flagged, target clean): {len(fp)}")
        if not fp.empty:
            display(fp[detail_cols].head(10))
        else:
            print("  None")

        print(f"\nFalse Negatives (scanner missed, target violation): {len(fn)}")
        if not fn.empty:
            display(fn[detail_cols].head(10))
        else:
            print("  None")